# 1. Setup & Load data

In [14]:
# Import libraries
import os
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Theme for the whole chart
pio.templates.default = "plotly_white"
COLORS = {"Adelie Penguin (Pygoscelis adeliae)": '#FF8C00', "Chinstrap penguin (Pygoscelis antarctica)": '#9932CC', "Gentoo penguin (Pygoscelis papua)": '#057076'}

In [15]:
# Load the preprocessed penguins dataset using a workspace-relative path
data_path = Path.cwd() / 'data' / 'penguins_preprocessed.csv'
if not data_path.exists():
    raise FileNotFoundError(f"Cannot find preprocessed dataset at {data_path}")

df = pd.read_csv(data_path)

# Check data
print(f"Cleaned dataset has: {df.shape[0]} rows and {df.shape[1]} columns")
display(df.describe())

Cleaned dataset has: 344 rows and 16 columns


,sample_number,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,delta_15_n,delta_13_c
count,344.000000,344.000000,344.000000,344.000000,344.000000,344.000000,344.000000
mean,63.151163,43.921930,17.151170,200.915205,4201.754386,8.733382,-25.686292
std,40.430199,5.443643,1.969027,14.020657,799.613058,0.540392,0.778770
min,1.000000,32.100000,13.100000,172.000000,2700.000000,7.632200,-27.018540
25%,29.000000,39.275000,15.600000,190.000000,3550.000000,8.307415,-26.285460
50%,58.000000,44.250000,17.300000,197.000000,4050.000000,8.687455,-25.793660
75%,95.250000,48.500000,18.700000,213.000000,4750.000000,9.136170,-25.089467
max,152.000000,59.600000,21.500000,231.000000,6300.000000,10.025440,-23.787670


# 2. EDA Visualizations (Directly run Notebook first)

In [16]:
# 1. Distribution: Weight by Species
fig1 = px.violin(df, y="body_mass_g", x="species", color="species", 
                 box=True, points="all", hover_data=df.columns,
                 title="Distribution of weights by species",
                 color_discrete_map=COLORS)
fig1.show()

In [17]:
# 2. Correlation: Correlation matrix of physical figures
corr = df.select_dtypes(include=[np.number]).corr()
fig2 = px.imshow(corr, text_auto=True, aspect="auto",
                 title="Correlation matrix of physical figures",
                 color_continuous_scale='RdBu_r')
fig2.show()

In [18]:
# 3. Scatter Matrix: Multivariable scatter matrix
# Scatter Matrix with horizontal Y-labels
fig3 = px.scatter_matrix(
    df, 
    dimensions=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"],
    color="species", 
    color_discrete_map=COLORS,
    title="Penguin Morphological Matrix",
    # Replacing "_"
    labels={col: col.replace('_', ' ').title() for col in df.columns}
)

fig3.update_traces(diagonal_visible=True)

# Modify y-axis for var names
fig3.update_yaxes(
    tickangle=0, 
    title_standoff=10, 
    automargin=True   
)

# Overall layout
fig3.update_layout(
    height=900, 
    width=1200,
    margin=dict(l=180, r=20, t=80, b=80), 
    autosize=True,
    title={
        'text': "Penguin Morphological Matrix",
        'y': 0.95,           
        'x': 0.5,            
        'xanchor': 'center', 
        'yanchor': 'top'
    }
)

fig3.show()

In [19]:
# 4. Species Count Bar Chart
# This visualization shows the sample size for each species in the dataset.
fig4 = px.histogram(
    df, 
    x="species", 
    color="species",
    color_discrete_map=COLORS,
    title="Total Count of Each Penguin Species",
    text_auto=True # Automatically displays the count
)

# Refine the layout and appearance
fig4.update_layout(
    title_x=0.5,
    xaxis_title="Penguin Species",
    yaxis_title="Number of Individuals",
    showlegend=False,          # Hide legend as X-axis already has the names
    plot_bgcolor="white",      # Clean white background
    bargap=0.3,                # Add space between bars (0.0 to 1.0)
    height=500
)

# Fine-tune the text labels and axis lines
fig4.update_traces(
    textposition='outside',    # Move numbers to the top of the bars
    textfont_size=14,
    marker_line_color='black', # Add a subtle border to bars
    marker_line_width=1
)

# Optional: Add subtle horizontal gridlines for easier reading
fig4.update_yaxes(showgrid=True, gridcolor='LightGray')

fig4.show()

In [20]:
# 5. Scatter Plot with Marginal Histograms
# Analyzing the relationship between Flipper Length and Bill Length with distribution density
try:
    import statsmodels.api as sm  # noqa: F401
    trendline_value = "ols"
except ImportError:
    trendline_value = None

fig5_kwargs = {
    'data_frame': df,
    'x': 'flipper_length_mm',
    'y': 'bill_length_mm',
    'color': 'species',
    'marginal_x': 'histogram',
    'marginal_y': 'violin',
    'color_discrete_map': COLORS,
    'title': 'Flipper Length vs. Bill Length (with Marginal Distributions)',
    'labels': {
        'flipper_length_mm': 'Flipper Length (mm)',
        'bill_length_mm': 'Bill Length (mm)'
    }
}
if trendline_value:
    fig5_kwargs['trendline'] = trendline_value

fig5 = px.scatter(**fig5_kwargs)
fig5.update_layout(
    title_x=0.5,
    height=600
)
fig5.show()

In [47]:
# 6. Multi-Feature Horizontal Bar Chart by Species and Sex
# Grouping data to get means for all 4 physical features
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df_multi = df.groupby(['species', 'sex'])[features].mean().reset_index()

# Melt the dataframe to convert features into a single column for faceting
df_melted = df_multi.melt(
    id_vars=["species", "sex"], 
    value_vars=features,
    var_name="feature", 
    value_name="average_value"
)

# Create the horizontal bar chart
fig6_custom = px.bar(
    df_melted,
    x="average_value",
    y="species",
    color="sex",
    orientation='h',
    barmode="group",
    facet_col="feature", 
    facet_col_wrap=2,   
    color_discrete_map={
        "Male": "#7f7f7f", 
        "Female": "#bcbd22"
    },
    title="Average Physical Features by Species and Sex",
    labels={
        "average_value": "Average Value",
        "species": "Species",
        "sex": "Sex",
        "feature": "Metric"
    }
)

# Formatting axes and layout
fig6_custom.update_yaxes(tickangle=0, ticksuffix="   ")
# Create a mapping dictionary for better display names with units
unit_mapping = {
    "bill_length_mm": "Bill Length (mm)",
    "bill_depth_mm": "Bill Depth (mm)",
    "flipper_length_mm": "Flipper Length (mm)",
    "body_mass_g": "Body Mass (g)"
}

# Update the annotations using the mapping
fig6_custom.for_each_annotation(lambda a: a.update(
    text=unit_mapping.get(a.text.split("=")[-1], a.text)
))
fig6_custom.update_xaxes(matches=None)

fig6_custom.update_layout(
    title_x=0.5,
    height=800,
    margin=dict(t=100, l=120, r=40, b=80),
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5)
)

fig6_custom.update_traces(
    texttemplate='%{x:.1f}', 
    textposition='outside'
)

fig6_custom.show()

In [22]:
# 7. Stacked Bar Chart: Species Composition by Island
fig7_stacked = px.histogram(
    df, 
    x="island", 
    color="species",
    color_discrete_map=COLORS,
    barmode="stack", # Bars are stacked on top of each other
    title="Species Distribution Across Islands (Stacked)",
    labels={
        "island": "Island",
        "species": "Species",
        "count": "Number of Individuals"
    },
    text_auto=True
)

fig7_stacked.update_layout(
    title_x=0.5,
    xaxis_title="Island Name",
    yaxis_title="Total Population",
    legend_title="Penguin Species",
    plot_bgcolor="white",
    height=600
)

fig7_stacked.update_traces(
    marker_line_color='white',
    marker_line_width=1
)

fig7_stacked.show()

In [38]:
# 8. Enhanced 3D Scatter Plot (Shifted Upwards)
# This increases the frame size and moves the 3D scene higher within the layout
fig8 = px.scatter_3d(
    df, x='bill_length_mm', y='flipper_length_mm', z='body_mass_g',
    color='species', color_discrete_map=COLORS,
    opacity=0.8, title="3D Morphological Clusters",
    labels={"bill_length_mm": "Bill Length (mm)", "flipper_length_mm": "Flipper Length (mm)", "body_mass_g": "Mass (g))"}
)

fig8.update_traces(marker=dict(size=4, line=dict(width=1, color='DarkSlateGrey')))

fig8.update_layout(
    height=950,      
    width=1100,         
    title_x=0.5,
    title_font_size=22,
    
    # The 'scene' controls the 3D c
    scene=dict(
        # domain y: [0.2, 1.0] pushes the cube 20% up
        domain=dict(y=[0.2, 1.0]), 
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
        xaxis=dict(gridcolor='white', zerolinecolor='white', showbackground=True, backgroundcolor="rgb(230, 230, 230)"),
        yaxis=dict(gridcolor='white', zerolinecolor='white', showbackground=True, backgroundcolor="rgb(230, 230, 230)"),
        zaxis=dict(gridcolor='white', zerolinecolor='white', showbackground=True, backgroundcolor="rgb(230, 230, 230)")
    ),
    
    # Adjusting margins
    margin=dict(l=0, r=0, b=50, t=100),
    
    # Legend placement
    legend=dict(
        yanchor="top",
        y=0.9,
        xanchor="center",
        x=0.05,
        bgcolor="rgba(255, 255, 255, 0.5)"
    )
)

fig8.show()

In [32]:
# 9. Isotope Scatter Plot: Diet & Habitat Analysis
# Analyzing the ecological niche of each species using Nitrogen and Carbon isotopes
fig9 = px.scatter(
    df, 
    x="delta_13_c", 
    y="delta_15_n", 
    color="species",
    color_discrete_map=COLORS,
    marginal_x="box", 
    marginal_y="box",
    title="Isotopic Niche of Penguin Species",
    labels={
        "delta_13_c": "Delta 13 C (o/oo) - Foraging Area",
        "delta_15_n": "Delta 15 N (o/oo) - Trophic Level"
    }
)

fig9.update_layout(
    title_x=0.5,
    height=700,
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)

fig9.update_xaxes(showgrid=True, gridcolor='LightGray')
fig9.update_yaxes(showgrid=True, gridcolor='LightGray')

fig9.show()

In [ ]:
# 10. Optimized Ternary Plot: Scaled Morphological Fingerprint
df_ternary = df.copy()

# Min-Max Scaling to spread the points across the triangle
for col in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]:
    df_ternary[col + "_scaled"] = (df_ternary[col] - df_ternary[col].min()) / (df_ternary[col].max() - df_ternary[col].min())

fig10_optimized = px.scatter_ternary(
    df_ternary, 
    a="bill_length_mm_scaled", 
    b="bill_depth_mm_scaled", 
    c="flipper_length_mm_scaled",
    color="species", 
    color_discrete_map=COLORS,
    size="body_mass_g",
    size_max=10,
    title="Relative Morphological Balance (Scaled Features)",
    labels={
        "bill_length_mm_scaled": "Bill Length",
        "bill_depth_mm_scaled": "Bill Depth",
        "flipper_length_mm_scaled": "Flipper Length"
    }
)

fig10_optimized.update_layout(title_x=0.5, height=700)
fig10_optimized.show()